In [16]:
#An analysis of relevant stock price responses to highly publicized events.
#Focused on the Stargate data center in Abilene, Texas, developed by Crusoe.

#Requires installation of the following, if necessary:
#pip install os yfinance numpy pandas matplotlib seaborn scipy statsmodels

In [17]:
#necessary imports
import os
import zipfile, io
import requests
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from scipy import stats as st
from statsmodels.regression.linear_model import OLS
from datetime import datetime, timedelta

#select tickers
equipmentTickers = [
    'ETN',
    'GEV',
    'PH'
]

powerTickers = [
    'VST',
    'CEG'
]

smrTickers = [
    'SMR',
    'OKLO',
    'NNE'
]

controlTickers = [
    'SPY'
]

allTickers = equipmentTickers + powerTickers + smrTickers + controlTickers

#events
events = [
    ("2024-06-05", "Crusoe begins construction"),
    ("2024-10-01", "Phase One build-to-suit construction announced"),
    ("2025-01-21", "White House announces Stargate at press conference"),
    ("2025-03-07", "OpenAI and Oracle announce planned GPU deployment"),
    ("2025-03-10", "Phase Two construction (six buildings) underway"),
    ("2025-05-22", "JPMorgan and Crusoe give substantial funds"),
    ("2025-09-23", "First building goes live"),
    ("2026-03-07", "Expansion canceled; winter outage revealed"),
]

eventsdf = pd.DataFrame(events, columns = ['date', 'eventDescription'])
eventsdf['date'] = pd.to_datetime(eventsdf['date'])
dates = eventsdf['date']

#download daily ticker prices (at close)
#begin one year before the first event on the list
downloadWindowStart = dates.min() - timedelta(days = 367)
downloadWindowEnd = datetime.today().strftime('%Y-%m-%d')

allPrices = yf.download(
    tickers = allTickers,
    start = downloadWindowStart,
    end = downloadWindowEnd,
    auto_adjust = True,
    progress = False
)["Close"]
allPrices.fillna(0, inplace = True)

allReturns = ((allPrices.shift(-1) / allPrices).shift(1)).tail(-1)
allLogReturns = np.log(allReturns)
allReturns.fillna(0, inplace = True)
allLogReturns.fillna(0, inplace = True)
allReturns.replace([np.inf, -np.inf], 0, inplace = True)
allLogReturns.replace([np.inf, -np.inf], 0, inplace = True)

allPricesDestination = "data/allPrices.csv"
allReturnsDestination = "data/allReturns.csv"
allLogReturnsDestination = "data/allLogReturns.csv"

allPrices.to_csv(allPricesDestination)
allReturns.to_csv(allReturnsDestination)
allLogReturns.to_csv(allLogReturnsDestination)

In [18]:
#compute returns for equal-weighted portfolios of each treatment category

ecolumns = []
le = len(equipmentTickers)
for etick in equipmentTickers:
    ecolumns.append(etick)
eReturns = pd.DataFrame(columns = ecolumns)
for ecol in ecolumns:
    eReturns[ecol] = allReturns[ecol]

eReturns.insert(
    loc = len(ecolumns),
    column = 'weightedReturn',
    value = 0.0
)
for etick in equipmentTickers:
    eReturns.insert(
        loc = len(ecolumns),
        column = f'{etick} weight',
        value = 0.0
    )
    
for tradingDay in eReturns.itertuples():
    enz = []
    for etick in equipmentTickers:
        zero = getattr(tradingDay, etick) == 0
        nonzero = getattr(tradingDay, etick) != 0
        if zero: eReturns.loc[tradingDay.Index, f'{etick} weight'] = 0
        if nonzero: enz.append(etick)

    for etick in enz:
        eweight = 1 / (len(enz))
        eReturns.loc[tradingDay.Index, f'{etick} weight'] = eweight
        eReturns.loc[tradingDay.Index, 'weightedReturn'] = (
            eReturns.loc[tradingDay.Index, 'weightedReturn'] +
            (eweight * eReturns.loc[tradingDay.Index, etick])
        )

eReturns.replace([np.inf, -np.inf], 0, inplace = True)
eReturnsDestination = "data/eReturns.csv"
eReturns.to_csv(eReturnsDestination)

                  
pcolumns = []
lp = len(powerTickers)
for ptick in powerTickers:
    pcolumns.append(ptick)
pReturns = pd.DataFrame(columns = pcolumns)
for pcol in pcolumns:
    pReturns[pcol] = allReturns[pcol]

pReturns.insert(
    loc = len(pcolumns),
    column = 'weightedReturn',
    value = 0.0
)
for ptick in powerTickers:
    pReturns.insert(
        loc = len(pcolumns),
        column = f'{ptick} weight',
        value = 0.0
    )
    
for tradingDay in pReturns.itertuples():
    pnz = []
    for ptick in powerTickers:
        zero = getattr(tradingDay, ptick) == 0
        nonzero = getattr(tradingDay, ptick) != 0
        if zero: pReturns.loc[tradingDay.Index, f'{ptick} weight'] = 0
        if nonzero: pnz.append(ptick)

    for ptick in pnz:
        pweight = 1 / (len(pnz))
        pReturns.loc[tradingDay.Index, f'{ptick} weight'] = pweight
        pReturns.loc[tradingDay.Index, 'weightedReturn'] = (
            pReturns.loc[tradingDay.Index, 'weightedReturn'] +
            (pweight * pReturns.loc[tradingDay.Index, ptick])
        )

pReturns.replace([np.inf, -np.inf], 0, inplace = True)
pReturnsDestination = "data/pReturns.csv"
pReturns.to_csv(pReturnsDestination)


scolumns = []
ls = len(smrTickers)
for stick in smrTickers:
    scolumns.append(stick)
sReturns = pd.DataFrame(columns = scolumns)
for scol in scolumns:
    sReturns[scol] = allReturns[scol]

sReturns.insert(
    loc = len(scolumns),
    column = 'weightedReturn',
    value = 0.0
)
for stick in smrTickers:
    sReturns.insert(
        loc = len(scolumns),
        column = f'{stick} weight',
        value = 0.0
    )
    
for tradingDay in sReturns.itertuples():
    snz = []
    for stick in smrTickers:
        zero = getattr(tradingDay, stick) == 0
        nonzero = getattr(tradingDay, stick) != 0
        if zero: sReturns.loc[tradingDay.Index, f'{stick} weight'] = 0
        if nonzero: snz.append(stick)

    for stick in snz:
        sweight = 1 / (len(snz))
        sReturns.loc[tradingDay.Index, f'{stick} weight'] = sweight
        sReturns.loc[tradingDay.Index, 'weightedReturn'] = (
            sReturns.loc[tradingDay.Index, 'weightedReturn'] +
            (sweight * sReturns.loc[tradingDay.Index, stick])
        )

sReturns.replace([np.inf, -np.inf], 0, inplace = True)
sReturnsDestination = "data/sReturns.csv"
sReturns.to_csv(sReturnsDestination)

In [69]:
#incorporate Fama-French three-factor model daily returns
#originally tried with pandas_datareader
#encountered compatibility issue that way
#used requests and zipfile instead

def getff(ffstart, ffend):
    url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip"

    response = requests.get(url)
    zf = zipfile.ZipFile(io.BytesIO(response.content)) 
    name = zf.namelist()[0]

    with zf.open(name) as f:
        
        #drop header rows
        #manually add encoding parameter just to be safe
        allff = pd.read_csv(f, skiprows = 4, index_col = 0, encoding = 'utf-8')
        
        #drop any remaining non-numeric rows (footer, etc)
        allff = allff[pd.to_numeric(allff['Mkt-RF'], errors = 'coerce').notna()]

        #change index column (the date) to datetime format
        allff.index = pd.to_datetime(allff.index, format = '%Y%m%d')

        #narrow it down to desired dates
        allff = allff.loc[ffstart:ffend]

        #change all factor returns (the whole dataframe) to float
        allff = allff.astype(float)

        #turn them into percentages in same format as our other returns
        allff = allff / 100
        allff = allff + 1

    return allff
    
ff3 = getff((downloadWindowStart + timedelta(days = 2)), downloadWindowEnd)

#add index name for cleanliness
ff3.index.name = 'Date'

#add the daily portfolio returns 
portfolioReturns = {
    'e': eReturns,
    'p': pReturns,
    's': sReturns
}

for pname, portfolio in portfolioReturns.items():
    ff3 = pd.merge(ff3, portfolio['weightedReturn'],
                   left_index = True, right_index = True, how = 'inner')
    i = len(ff3.columns)
    ff3.columns.values[i-1] = f'{pname}Weighted'

print(ff3.columns.tolist())


['Mkt-RF', 'SMB', 'HML', 'RF', 'eWeighted', 'pWeighted', 'sWeighted']


In [77]:
#using the columns of the ff3 dataframe, find beta
ff3['eExcess'] = ff3['eWeighted'] - ff3['RF']
ff3['pExcess'] = ff3['pWeighted'] - ff3['RF']
ff3['sExcess'] = ff3['sWeighted'] - ff3['RF']
print(ff3.head(10))

x = ff3[['Mkt-RF', 'SMB', 'HML']]
#ff3.rename(columns = {'const': 'alpha'})

            Mkt-RF     SMB     HML      RF  eWeighted  pWeighted  sWeighted  \
Date                                                                          
2023-06-06  1.0046  1.0213  1.0135  1.0002   1.013172   1.008242   1.002182   
2023-06-07  0.9964  1.0136  1.0228  1.0002   1.031952   1.015729   1.026091   
2023-06-08  1.0059  0.9951  0.9884  1.0002   0.993600   1.001447   0.997234   
2023-06-09  1.0003  0.9929  0.9959  1.0002   0.999037   1.005363   0.979429   
2023-06-12  1.0094  1.0004  0.9859  1.0002   1.009460   1.000048   1.024663   
2023-06-13  1.0074  1.0059  1.0006  1.0002   1.024811   1.006715   1.013645   
2023-06-14  0.9997  0.9910  0.9908  1.0002   0.997039   0.992622   0.988973   
2023-06-15  1.0122  0.9963  1.0009  1.0002   1.018721   1.011875   0.991300   
2023-06-16  0.9955  0.9945  1.0022  1.0002   0.979099   0.991621   1.003464   
2023-06-20  0.9952  1.0043  0.9931  1.0002   1.011490   0.997449   0.983240   

             eExcess   pExcess   sExcess  
Date    